# Tabelas de tempo — configurações escolhidas na validação

Lê os resultados de `measure_time_chosen.py` (`results_time_chosen/<conjunto>/<binarização>/seed<k>.jsonl`) e imprime, para cada conjunto, uma tabela LaTeX com uma linha por modelo:

| coluna | o que é |
|---|---|
| Training, Inference | one-shot ou por épocas; bleaching iterativo, DAB ou rDAB |
| Epoch | por semente, na ordem das sementes: `bestEpoch`, a época do modelo entregue (0 = a primeira, como no relatório do `trainEpochs`); com `EPOCH_FIELD = "epochsRun"`, as épocas rodadas |
| Train (s) | `train` / `trainEpochs`, este com a validação de cada época (early stopping) |
| Test inference (s) | `classify` do teste, uma thread |
| Acc. (%) | acurácia de teste |

Média $\pm$ desvio-padrão amostral (ddof = 1) entre as sementes; com uma semente só a média. A configuração de cada semente é a de melhor validação, como em `modes_detail` do `benchmark_table.ipynb`.

Tempos com 2 casas; abaixo de 0,1 s, com as casas necessárias para 2 algarismos significativos da média (`TIME_DECIMALS = 2` fixa 2 casas em tudo).

In [1]:
import json
import math
from pathlib import Path

import numpy as np

# the notebook lives in experiments/, but also works when started from the repository root
HERE = Path.cwd() if (Path.cwd() / "measure_time_chosen.py").exists() else Path.cwd() / "experiments"
RESULTS_DIR = HERE / "results_time_chosen"
ENCODING = "distributive"  # or "thermometer"
EPOCH_FIELD = "bestEpoch"  # "bestEpoch": epoch of the delivered model (0 = first); "epochsRun": epochs run
TIME_DECIMALS = None  # None: 2, or 2 significant digits below 0.1 s; an int fixes the decimals
ACC_DECIMALS = 1

# model of measure_time_chosen.py -> (Training, Inference), in the order of the table
ROWS = [
    ("wisard-bleaching", "One-shot", "Iter."),
    ("wisard-DAB", "One-shot", "DAB"),
    ("wisard-rDAB", "One-shot", "$r$DAB"),
    ("epochs-bleaching", "Epoch-based", "Iter."),
    ("epochs-DAB", "Epoch-based", "DAB"),
    ("epochs-rDAB", "Epoch-based", "$r$DAB"),
]

In [2]:
def load(results_dir, encoding):
    """{dataset: {model: {seed: result}}}; when a model was measured twice in a seed, the last one"""
    out = {}
    for path in sorted(results_dir.glob("*/%s/seed*.jsonl" % encoding)):
        dataset = path.parts[-3]
        with open(path) as f:
            for r in map(json.loads, f):
                if "error" not in r:
                    out.setdefault(dataset, {}).setdefault(r["model"], {})[r["seed"]] = r
    return out


def decimals(values, fixed):
    if fixed is not None:
        return fixed
    mean = abs(float(np.mean(values)))
    if mean >= 0.1 or mean == 0:
        return 2
    return min(6, 1 - math.floor(math.log10(mean)))  # 2 significant digits


def pm(values, digits):
    """mean ± sample std, in LaTeX (only the mean with one value)"""
    if len(values) == 1:
        return "%.*f" % (digits, values[0])
    return "%.*f$\\pm$%.*f" % (digits, np.mean(values), digits, np.std(values, ddof=1))


def latex_table(dataset, models):
    seeds = sorted({s for by_seed in models.values() for s in by_seed})
    lines = ["%% %s (%s), seeds %s" % (dataset, ENCODING, ", ".join(map(str, seeds))),
             r"\begin{tabular}{lllccc}", r"\toprule",
             r"Training & Inference & Epoch    & Train (s) & Test inference (s) & Acc. (\%) \\", r"\midrule"]
    for model, training, inference in ROWS:
        by_seed = models.get(model, {})
        if not by_seed:
            lines.append("%% %s: no results" % model)
            continue
        results = [by_seed[s] for s in sorted(by_seed)]
        if sorted(by_seed) != seeds:
            lines.append("%% %s: only seeds %s" % (model, ", ".join(map(str, sorted(by_seed)))))
        epochs = ", ".join(str(r[EPOCH_FIELD]) for r in results) if model.startswith("epochs") else "--"
        train = [r["secondsTrain"] for r in results]
        infer = [r["secondsInfer"] for r in results]
        acc = [100 * r["accTest"] for r in results]
        lines.append("%-11s & %-6s & %s & %s & %s & %s \\\\" % (
            training, inference, epochs, pm(train, decimals(train, TIME_DECIMALS)),
            pm(infer, decimals(infer, TIME_DECIMALS)), pm(acc, ACC_DECIMALS)))
    lines += [r"\bottomrule", r"\end{tabular}"]
    return "\n".join(lines)


data = load(RESULTS_DIR, ENCODING)
print("%d datasets in %s (%s): %s" % (len(data), RESULTS_DIR, ENCODING, ", ".join(sorted(data))))

15 datasets in c:\Users\yasmin.romeiro\OneDrive - RNP\Documentos\Pessoal - Yasmin\Mestrado\wisardpkg\experiments\results_time_chosen (distributive): australian, blood, christine, fashion-mnist, higgs, iris, jasmine, jsc, miniboone, mnist, nomao, phoneme, segment, skin-seg, sylvine


In [3]:
for dataset in sorted(data):
    print(latex_table(dataset, data[dataset]))
    print()

% australian (distributive), seeds 0, 1, 2, 3, 4
\begin{tabular}{lllccc}
\toprule
Training & Inference & Epoch    & Train (s) & Test inference (s) & Acc. (\%) \\
\midrule
One-shot    & Iter.  & -- & 0.00034$\pm$0.00014 & 0.00023$\pm$0.00005 & 83.3$\pm$3.1 \\
One-shot    & DAB    & -- & 0.00040$\pm$0.00034 & 0.00027$\pm$0.00014 & 84.2$\pm$3.3 \\
One-shot    & $r$DAB & -- & 0.00029$\pm$0.00009 & 0.00019$\pm$0.00002 & 84.2$\pm$3.4 \\
Epoch-based & Iter.  & 2, 6, 0, 2, 0 & 0.0055$\pm$0.0025 & 0.00026$\pm$0.00005 & 86.7$\pm$3.4 \\
Epoch-based & DAB    & 0, 9, 3, 1, 8 & 0.0070$\pm$0.0034 & 0.00026$\pm$0.00007 & 85.8$\pm$4.6 \\
Epoch-based & $r$DAB & 0, 1, 0, 0, 0 & 0.0043$\pm$0.0009 & 0.00021$\pm$0.00005 & 86.0$\pm$4.3 \\
\bottomrule
\end{tabular}

% blood (distributive), seeds 0, 1, 2, 3, 4
\begin{tabular}{lllccc}
\toprule
Training & Inference & Epoch    & Train (s) & Test inference (s) & Acc. (\%) \\
\midrule
One-shot    & Iter.  & -- & 0.00029$\pm$0.00016 & 0.00030$\pm$0.00017 & 75.5$\pm$